# 01 — Steam Data Preparation

## Objective

This notebook prepares the Steam videogame dataset for large-scale exploratory analysis using **PySpark in Databricks**.

The objectives are to:

- load the official Steam dataset directly from Amazon S3;
- inspect the Spark schema and nested JSON structure;
- identify the fields required for the business analysis;
- assess missing values, duplicates, malformed values, and unusual observations;
- clean and standardize the selected analytical variables using Spark-native operations;
- derive reusable features for market, genre, and platform analysis;
- validate the resulting analytical dataset before performing exploratory analysis.

The prepared data will be used in the following notebooks for **Steam market analysis**, **genre analysis**, **platform analysis**, and the final business recommendations for Ubisoft.

---

## Business Context

Ubisoft wants to better understand the videogame ecosystem on Steam before releasing a new videogame.

The main business question is:

> **What does the Steam videogame market look like, and what factors appear associated with successful or popular games?**

The project investigates several dimensions of the Steam marketplace, including publishers, releases over time, prices and discounts, player reviews, languages, age requirements, genres, and supported platforms.

Where direct sales or revenue information is unavailable, variables such as review volume or ownership information may be considered as **popularity or commercial-attractiveness proxies**. These proxies will be interpreted cautiously and will not be presented as direct measures of sales or profitability.

---

## Methodology

This project follows a **Big Data exploratory data analysis** approach using PySpark.

The main data-processing pipeline is:

**Amazon S3 JSON → Spark DataFrame → schema inspection → selective nested-field extraction → data cleaning → derived features → analytical Spark DataFrames → distributed aggregations → Databricks visualizations**

The dataset is semi-structured and contains nested fields. Therefore, its actual Spark schema will be inspected before selecting, flattening, or exploding any fields.

The analysis follows a **Spark-first methodology**:

- meaningful transformations are performed using Spark DataFrame operations;
- nested fields are extracted only when required by the analytical questions;
- arrays are exploded only when a game-level structure must be transformed into a category-level structure;
- Spark-native functions are preferred over Python loops and user-defined functions;
- unnecessary `collect()` operations and full-data conversions to pandas are avoided;
- cleaning decisions are documented and applied reproducibly;
- legitimate extreme observations are distinguished from data-quality errors.

This approach preserves distributed processing while keeping the workflow concise, reproducible, and aligned with the Big Data objectives of the project.

---

## Execution Environment

The analysis is executed in **Databricks**, which provides the Spark execution environment used for distributed data processing.

The source dataset remains in Amazon S3 and is not stored in the GitHub repository.

Completed Databricks notebooks will be exported to the project's `notebooks/` directory, while selected visualization screenshots will be stored under `outputs/figures/` for the final GitHub presentation.

---

## Notebook Structure

1. **Environment and Data Loading**
   - project configuration
   - load the Steam dataset from Amazon S3
   - initial dataset inspection

2. **Schema and Nested Structure Analysis**
   - inspect the nested JSON schema
   - select the analytical fields
   - validate the extracted schema
   - verify identifier consistency

3. **Data Quality Assessment**
   - dataset size and identifier uniqueness
   - duplicate records
   - missing values
   - empty-string values
   - basic numerical integrity checks
   - summarize the main data-quality findings

4. **Data Cleaning and Feature Preparation**
   - core cleaning and type conversion
   - release-date preparation
   - price, discount, and free-to-play preparation
   - review and popularity features
   - retain videogame applications only
   - ownership-range preparation
   - genre, language, and platform features
   - final analytical dataset validation

5. **Prepared Dataset Summary**
   - summarize the final cleaned dataset
   - document remaining missing information and limitations
   - identify the analytical dimensions available for subsequent EDA


# 1. Environment and Data Loading

This section initializes the PySpark environment and loads the Steam videogame dataset for distributed analysis.

The dataset is stored as a **semi-structured JSON file on Amazon S3** and is read directly into a Spark DataFrame using Databricks. Keeping the source data in S3 avoids unnecessary local storage and allows the subsequent transformations and aggregations to remain within the Spark ecosystem.

The initial inspection focuses on:

* confirming the Spark execution environment and source path;
* loading the JSON dataset into a Spark DataFrame;
* identifying the top-level columns;
* inspecting a small sample of records before performing any transformations.

At this stage, the dataset is kept in its original structure. No cleaning or flattening is performed until the nested schema has been examined in the following section.


In [0]:
# PySpark imports
# ---------------------------------------------------------------------------

from pyspark.sql import functions as F


# Project configuration
# ---------------------------------------------------------------------------

STEAM_DATA_PATH = (
    "s3://full-stack-bigdata-datasets/"
    "Big_Data/Project_Steam/steam_game_output.json"
)


# Environment information
# ---------------------------------------------------------------------------

print(f"Spark version: {spark.version}")
print(f"Steam dataset: {STEAM_DATA_PATH}")

Spark version: 4.2.0
Steam dataset: s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json


In [0]:
# Load the Steam JSON dataset from Amazon S3
# ---------------------------------------------------------------------------

steam_raw_df = (
    spark.read
    .format("json")
    .load(STEAM_DATA_PATH)
)


# Confirm the resulting Spark DataFrame
# ---------------------------------------------------------------------------

print(type(steam_raw_df))

<class 'pyspark.sql.connect.dataframe.DataFrame'>


In [0]:
# Inspect the complete nested Spark schema
# ---------------------------------------------------------------------------

steam_raw_df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:
# Display the top-level structure
# ---------------------------------------------------------------------------

print(f"Number of top-level columns: {len(steam_raw_df.columns)}")

steam_raw_df.columns

Number of top-level columns: 2


['data', 'id']

In [0]:
# Inspect a small sample without collecting the dataset locally
# ---------------------------------------------------------------------------

display(steam_raw_df.limit(5))

data,id
"List(10, List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP), 13990, Valve, 0, Action, https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513, 999, English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean, Counter-Strike, 5199, 10,000,000 .. 20,000,000, List(true, true, true), 201215, 999, Valve, 2000/11/1, 0, Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role., List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192), game, )",10
"List(1000000, List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud), 0, IndigoBlue Game Studio, 0, Action, Adventure, Indie, https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048, 999, English, Korean, Simplified Chinese, ASCENXION, 5, 0 .. 20,000, List(false, false, true), 27, 999, PsychoFlux Entertainment, 2021/05/14, 0, ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, 

## 2. Schema and Nested Structure Analysis

The initial inspection shows that the dataset contains two top-level fields:

- `id`, which identifies the Steam application;
- `data`, a nested struct containing the game attributes.

Most variables required for the analysis are therefore stored inside the `data` struct rather than as top-level columns.

The schema also contains different structural representations that require different processing strategies:

- `categories` is an array of strings;
- `platforms` is a nested struct containing Boolean indicators for Windows, macOS, and Linux;
- `genre` and `languages` are stored as delimited strings rather than arrays;
- price, discount, age, and date variables are initially represented as strings;
- `owners` is represented as an ownership range rather than an exact numerical value;
- `tags` is a large nested struct containing many individual Steam tags.

The analysis will therefore use **selective flattening**: only fields that support the defined business questions will be extracted from the nested structure. The large `tags` struct and other non-essential attributes will not be flattened, avoiding unnecessary complexity and processing.

### 2.1 Analytical Field Selection

A compact game-level DataFrame is created by extracting the variables required for the subsequent market, genre, and platform analyses.

At this stage, the objective is only to simplify the nested structure and assign clear analytical column names. Data types and values are not cleaned yet.

The selected variables cover:

- game identification;
- publisher and developer information;
- release timing;
- pricing and discounts;
- player reviews and concurrent users;
- ownership ranges;
- genres and languages;
- age requirements;
- categories;
- platform availability.

The original `tags` struct, image URL, website, and description fields are excluded because they are not required to answer the selected business questions.

In [0]:
# Extract the analytical fields from the nested data struct
# ---------------------------------------------------------------------------

steam_selected_df = steam_raw_df.select(
    F.col("id").alias("game_id"),
    F.col("data.appid").alias("appid"),
    F.col("data.name").alias("name"),
    F.col("data.type").alias("type"),
    F.col("data.publisher").alias("publisher"),
    F.col("data.developer").alias("developer"),
    F.col("data.release_date").alias("release_date"),
    F.col("data.initialprice").alias("initial_price_raw"),
    F.col("data.price").alias("price_raw"),
    F.col("data.discount").alias("discount_raw"),
    F.col("data.positive").alias("positive_reviews"),
    F.col("data.negative").alias("negative_reviews"),
    F.col("data.ccu").alias("concurrent_users"),
    F.col("data.owners").alias("owners_raw"),
    F.col("data.genre").alias("genres_raw"),
    F.col("data.languages").alias("languages_raw"),
    F.col("data.required_age").alias("required_age_raw"),
    F.col("data.categories").alias("categories"),
    F.col("data.platforms.windows").alias("supports_windows"),
    F.col("data.platforms.mac").alias("supports_mac"),
    F.col("data.platforms.linux").alias("supports_linux"),
)

### 2.2 Extracted Schema Validation

The schema of the selected DataFrame is inspected to verify that the nested fields were extracted correctly and that their original Spark data types are preserved.

This validation is performed before cleaning so that subsequent transformations can be based on the actual source representation of each variable.

In [0]:
# Validate the schema after selective nested-field extraction
# ---------------------------------------------------------------------------

steam_selected_df.printSchema()

root
 |-- game_id: string (nullable = true)
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- type: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- initial_price_raw: string (nullable = true)
 |-- price_raw: string (nullable = true)
 |-- discount_raw: string (nullable = true)
 |-- positive_reviews: long (nullable = true)
 |-- negative_reviews: long (nullable = true)
 |-- concurrent_users: long (nullable = true)
 |-- owners_raw: string (nullable = true)
 |-- genres_raw: string (nullable = true)
 |-- languages_raw: string (nullable = true)
 |-- required_age_raw: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- supports_windows: boolean (nullable = true)
 |-- supports_mac: boolean (nullable = true)
 |-- supports_linux: boolean (nullable = true)



In [0]:
# Inspect representative records from the selected game-level DataFrame
# ---------------------------------------------------------------------------

display(steam_selected_df.limit(10))

game_id,appid,name,type,publisher,developer,release_date,initial_price_raw,price_raw,discount_raw,positive_reviews,negative_reviews,concurrent_users,owners_raw,genres_raw,languages_raw,required_age_raw,categories,supports_windows,supports_mac,supports_linux
10,10,Counter-Strike,game,Valve,Valve,2000/11/1,999,999,0,201215,5199,13990,"10,000,000 .. 20,000,000",Action,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",0,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",true,true,true
1000000,1000000,ASCENXION,game,PsychoFlux Entertainment,IndigoBlue Game Studio,2021/05/14,999,999,0,27,5,0,"0 .. 20,000","Action, Adventure, Indie","English, Korean, Simplified Chinese",0,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)",true,false,false
1000010,1000010,Crown Trick,game,"Team17, NEXT Studios",NEXT Studios,2020/10/16,1999,599,70,4032,646,99,"200,000 .. 500,000","Adventure, Indie, RPG, Strategy","Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil",0,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)",true,false,false
1000030,1000030,"Cook, Serve, Delicious! 3?!",game,Vertigo Gaming Inc.,Vertigo Gaming Inc.,2020/10/14,1999,1999,0,1575,115,76,"100,000 .. 200,000","Action, Indie, Simulation, Strategy",English,0,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)",true,true,false
1000040,1000040,细胞战争,game,DoubleC Games,DoubleC Games,2019/03/30,199,199,0,0,1,0,"0 .. 20,000","Action, Casual, Indie, Simulation",Simplified Chinese,0,List(Single-player),true,false,false
1000080,1000080,Zengeon,game,2P Games,IndieLeague Studio,2019/06/24,1999,799,60,1018,462,3,"100,000 .. 200,000","Action, Adventure, Indie, RPG","Simplified Chinese, English, Traditional Chinese, Japanese, Korean",0,"List(Multi-player, Single-player, Steam Achievements, Full controller support, Steam Trading Cards)",true,true,false
1000100,1000100,干支セトラ 陽ノ卷｜干支etc. 陽之卷,game,Starship Studio,七月九日,2019/01/24,1299,1299,0,18,6,0,"0 .. 20,000","Adventure, Indie, RPG, Strategy","Japanese, Simplified Chinese, Traditional Chinese",0,"List(Single-player, Steam Achievements, Steam Cloud)",true,false,false
1000110,1000110,Jumping Master(跳跳大咖),game,重庆环游者网络科技,重庆环游者网络科技,2019/04/8,0,0,0,50,34,0,"20,000 .. 50,000","Action, Adventure, Casual, Free to Play, Massively Multiplayer","English, Simplified Chinese, Traditional Chinese",0,"List(Multi-player, Single-player, Co-op, Online PvP, Online Co-op, PvP)",true,false,false
1000130,1000130,Cube Defender,game,Simon Codrington,Simon Codrington,2019/01/6,299,299,0,6,0,0,"0 .. 20,000","Casual, Indie",English,0,"List(Single-player, Steam Achievements, Steam Leaderboards)",true,true,false
1000280,1000280,Tower of Origin2-Worm's Nest,game,Villain Role,Villain Role,2021/09/9,1399,1399,0,32,12,0,"0 .. 20,000","Indie, RPG","English, Simplified Chinese, Traditional Chinese",0,List(Single-player),true,false,false


### 2.3 Identifier Consistency

The source contains both a top-level `id` and a nested `appid`.

The sample records suggest that these fields may represent the same Steam application identifier. Their consistency is verified across the dataset before deciding whether both variables need to be retained.

In [0]:
# Check whether the two source identifiers disagree
# ---------------------------------------------------------------------------

identifier_check_df = steam_selected_df.agg(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(
            F.col("game_id").isNull() | F.col("appid").isNull(),
            1,
        ).otherwise(0)
    ).alias("rows_with_missing_identifier"),
    F.sum(
        F.when(
            F.col("game_id") != F.col("appid"),
            1,
        ).otherwise(0)
    ).alias("identifier_mismatches"),
)

display(identifier_check_df)

total_rows,rows_with_missing_identifier,identifier_mismatches
55691,0,0


## 3. Data Quality Assessment

Before applying any cleaning rules, the selected analytical variables are assessed for completeness and structural consistency.

The purpose of this section is to distinguish genuine data-quality issues from valid business observations. In particular, the analysis examines:

- total number of records;
- uniqueness of the game identifier;
- duplicate records;
- missing values;
- empty strings in textual fields;
- completeness of platform information;
- basic validity of numerical review and activity variables.

No records are removed at this stage. The objective is first to quantify the quality issues and use the evidence to define explicit cleaning rules.

### 3.1 Dataset Size and Identifier Uniqueness

The number of records is compared with the number of distinct Steam application identifiers.

This verifies whether the dataset contains multiple rows for the same game before any transformations are performed.

In [0]:
# Assess dataset size and identifier uniqueness
# ---------------------------------------------------------------------------

dataset_summary_df = steam_selected_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("appid").alias("distinct_appids"),
)

display(dataset_summary_df)

total_rows,distinct_appids
55691,55691


### 3.2 Duplicate Records

Duplicate application identifiers are inspected separately.

If each `appid` represents one Steam application, every identifier should appear only once in the game-level dataset.

In [0]:
# Identify duplicated Steam application identifiers
# ---------------------------------------------------------------------------

duplicate_appids_df = (
    steam_selected_df
    .groupBy("appid")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

duplicate_summary_df = duplicate_appids_df.agg(
    F.count("*").alias("duplicated_appids")
)

display(duplicate_summary_df)

duplicated_appids
0


### 3.3 Missing Values

Missing values are quantified across the analytical variables.

For string variables, both Spark `null` values and empty strings are relevant. They are assessed separately because an empty string is not technically equivalent to a null value but may still represent missing information.

The results will be used to determine field-specific cleaning decisions rather than applying a single blanket rule across the dataset.

In [0]:
# Count null values across all selected analytical columns
# ---------------------------------------------------------------------------

null_counts_df = steam_selected_df.agg(
    *[
        F.sum(F.col(column).isNull().cast("int")).alias(column)
        for column in steam_selected_df.columns
    ]
)

display(null_counts_df)

game_id,appid,name,type,publisher,developer,release_date,initial_price_raw,price_raw,discount_raw,positive_reviews,negative_reviews,concurrent_users,owners_raw,genres_raw,languages_raw,required_age_raw,categories,supports_windows,supports_mac,supports_linux
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Compute dataset size once for percentage calculations
# ---------------------------------------------------------------------------

total_rows = steam_selected_df.count()


# Reshape null counts for easier interpretation
# ---------------------------------------------------------------------------

null_counts_long_df = (
    null_counts_df
    .select(
        F.explode(
            F.array(
                *[
                    F.struct(
                        F.lit(column).alias("column"),
                        F.col(column).alias("null_count"),
                    )
                    for column in steam_selected_df.columns
                ]
            )
        ).alias("missing")
    )
    .select("missing.*")
    .withColumn(
        "null_percentage",
        F.round(
            F.col("null_count") / F.lit(total_rows) * 100,
            2,
        ),
    )
    .orderBy(F.desc("null_count"))
)

display(null_counts_long_df)

column,null_count,null_percentage
supports_mac,0,0.0
owners_raw,0,0.0
required_age_raw,0,0.0
supports_windows,0,0.0
positive_reviews,0,0.0
concurrent_users,0,0.0
discount_raw,0,0.0
languages_raw,0,0.0
categories,0,0.0
initial_price_raw,0,0.0


### 3.4 Empty String Values

Several source fields are stored as strings and may contain empty values instead of explicit nulls.

Empty strings are therefore counted separately for the main textual and encoded numerical fields before defining the cleaning rules.

In [0]:
# Count empty strings in relevant string columns
# ---------------------------------------------------------------------------

string_columns = [
    "name",
    "type",
    "publisher",
    "developer",
    "release_date",
    "initial_price_raw",
    "price_raw",
    "discount_raw",
    "owners_raw",
    "genres_raw",
    "languages_raw",
    "required_age_raw",
]

empty_string_counts_df = steam_selected_df.agg(
    *[
        F.sum(
            F.when(
                F.trim(F.col(column)) == "",
                1,
            ).otherwise(0)
        ).alias(column)
        for column in string_columns
    ]
)

display(empty_string_counts_df)

name,type,publisher,developer,release_date,initial_price_raw,price_raw,discount_raw,owners_raw,genres_raw,languages_raw,required_age_raw
0,0,134,127,99,0,0,0,0,161,11,0


### 3.5 Basic Numerical Integrity Checks

The numerical variables that are already correctly typed by Spark are checked for impossible negative values.

Negative review counts or concurrent-user counts would indicate invalid source records rather than legitimate business observations.

In [0]:
# Check for impossible negative values in numerical source variables
# ---------------------------------------------------------------------------

numerical_integrity_df = steam_selected_df.agg(
    F.sum(
        F.when(F.col("positive_reviews") < 0, 1).otherwise(0)
    ).alias("negative_positive_review_counts"),

    F.sum(
        F.when(F.col("negative_reviews") < 0, 1).otherwise(0)
    ).alias("negative_negative_review_counts"),

    F.sum(
        F.when(F.col("concurrent_users") < 0, 1).otherwise(0)
    ).alias("negative_concurrent_user_counts"),
)

display(numerical_integrity_df)

negative_positive_review_counts,negative_negative_review_counts,negative_concurrent_user_counts
0,0,0


### 3.6 Data Quality Assessment — Key Findings

The initial quality assessment indicates that the dataset is structurally complete and does not contain major integrity issues.

**Identifier integrity.** The dataset contains **55,691 records and 55,691 distinct `appid` values**, confirming that each Steam application appears exactly once. No duplicated application identifiers were detected. In addition, the previous identifier consistency check found no missing identifiers and no disagreement between the top-level `game_id` and nested `appid`. The numeric `appid` can therefore be retained as the canonical game identifier, while the redundant `game_id` field can be removed during cleaning.

**Missing values.** None of the selected analytical fields contain Spark `null` values. However, the empty-string analysis shows that absence of information is sometimes encoded as an empty string rather than as a null value. Empty values occur in **161 genre records, 134 publisher records, 127 developer records, 99 release dates, and 11 language records**. All other inspected string variables contain no empty strings.

This distinction is important: relying exclusively on conventional null detection would incorrectly suggest that the selected dataset is fully complete. The cleaning process must therefore account for both null and empty-string representations of missing information.

**Numerical integrity.** No impossible negative values were detected in the existing numerical variables: positive reviews, negative reviews, and concurrent users all contain zero negative observations. Zero values will not automatically be treated as missing, since they can represent legitimate observations such as a game with no recorded reviews or no concurrent users at the time represented by the dataset.

Overall, the dataset requires **targeted rather than aggressive cleaning**. The main preparation tasks are to standardize empty strings, remove the redundant identifier, convert encoded string variables to appropriate analytical types, parse release dates, interpret prices correctly, and derive reusable analytical features. Records will only be excluded from specific analyses when the required variable is unavailable or invalid, rather than being removed globally without justification.

## 4. Data Cleaning and Feature Preparation

Based on the data-quality assessment, the dataset requires targeted rather than extensive cleaning.

The preparation strategy focuses only on transformations required for the subsequent exploratory analyses:

1. remove the redundant source identifier and standardize empty strings;
2. convert encoded numerical variables to appropriate analytical types;
3. parse release dates and derive the release year;
4. create a small set of reusable features for reviews, ownership, genres, languages, and platforms.

The objective is to create one reliable game-level analytical DataFrame while preserving as many valid observations as possible. Missing information is handled at the variable or analysis level rather than by globally removing incomplete records.

### 4.1 Core Cleaning and Type Conversion

The numeric `appid` is retained as the unique game identifier because it is complete, unique, and fully consistent with the redundant top-level `game_id`.

Empty strings in the fields identified during the quality assessment are standardized as null values. This creates a consistent representation of missing information without removing the corresponding games.

Discount and age variables are also converted from their source string representation to numerical types for subsequent analysis.

In [0]:
# Create the cleaned game-level DataFrame
# ---------------------------------------------------------------------------

steam_clean_df = (
    steam_selected_df
    .drop("game_id")
    
    # Standardize empty strings as missing values
    .withColumn(
        "publisher",
        F.when(F.trim(F.col("publisher")) == "", None)
        .otherwise(F.trim(F.col("publisher")))
    )
    .withColumn(
        "developer",
        F.when(F.trim(F.col("developer")) == "", None)
        .otherwise(F.trim(F.col("developer")))
    )
    .withColumn(
        "release_date",
        F.when(F.trim(F.col("release_date")) == "", None)
        .otherwise(F.trim(F.col("release_date")))
    )
    .withColumn(
        "genres_raw",
        F.when(F.trim(F.col("genres_raw")) == "", None)
        .otherwise(F.trim(F.col("genres_raw")))
    )
    .withColumn(
        "languages_raw",
        F.when(F.trim(F.col("languages_raw")) == "", None)
        .otherwise(F.trim(F.col("languages_raw")))
    )
    
# Convert simple encoded numerical fields
# ---------------------------------------------------------------------------

.withColumn(
    "discount_pct",
    F.col("discount_raw").cast("double")
)
.withColumn(
    "required_age",
    F.expr("try_cast(required_age_raw as int)")
)
)

In [0]:
# Validate the first cleaning transformations
# ---------------------------------------------------------------------------

steam_clean_df.select(
    "appid",
    "name",
    "publisher",
    "developer",
    "release_date",
    "discount_pct",
    "required_age",
    "genres_raw",
    "languages_raw",
).printSchema()

root
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- required_age: integer (nullable = true)
 |-- genres_raw: string (nullable = true)
 |-- languages_raw: string (nullable = true)



In [0]:
# Verify standardized missing values and numeric conversions
# ---------------------------------------------------------------------------

core_cleaning_check_df = steam_clean_df.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.col("publisher").isNull().cast("int")).alias("missing_publisher"),
    F.sum(F.col("developer").isNull().cast("int")).alias("missing_developer"),
    F.sum(F.col("release_date").isNull().cast("int")).alias("missing_release_date"),
    F.sum(F.col("genres_raw").isNull().cast("int")).alias("missing_genres"),
    F.sum(F.col("languages_raw").isNull().cast("int")).alias("missing_languages"),
    F.sum(F.col("discount_pct").isNull().cast("int")).alias("invalid_discount"),
    F.sum(F.col("required_age").isNull().cast("int")).alias("invalid_required_age"),
)

display(core_cleaning_check_df)

total_rows,missing_publisher,missing_developer,missing_release_date,missing_genres,missing_languages,invalid_discount,invalid_required_age
55691,134,127,99,161,11,0,3


### 4.1 Cleaning Validation — Key Findings

The first cleaning step preserved all **55,691 records** while standardizing the small number of missing textual values identified during the quality assessment.

The resulting missing-value counts are consistent with the earlier diagnostics:

- 134 missing publishers;
- 127 missing developers;
- 99 missing release dates;
- 161 missing genre values;
- 11 missing language values.

The discount field converted successfully for all records, confirming that its source representation is numerically consistent.

The age-restriction field required tolerant parsing because a small number of records contain non-numeric rating labels. Only **3 records** could not be converted to a numerical age value. Given the very small proportion of affected observations, these values will remain null in the numerical age feature rather than introducing additional parsing complexity.

Overall, the cleaning process remains deliberately conservative: valid records are preserved and transformations are applied only where required for the subsequent analysis.

### 4.2 Release Date Preparation

The release date is required for the analysis of Steam publishing activity over time.

The source variable is stored as a string in `yyyy/M/d` format, with single-digit months and days possible. It is therefore converted to a Spark date type before deriving the release year.

Only two features are required for the planned analysis:

- a parsed release date;
- the release year.

Additional calendar features such as month, weekday, or quarter are not created because they are not required by the business questions.

In [0]:
# Parse release dates safely and derive release year
# ---------------------------------------------------------------------------

steam_clean_df = (
    steam_clean_df
    .withColumn(
        "release_date_parsed",
        F.to_date(
            F.try_to_timestamp(
                F.col("release_date"),
                F.lit("yyyy/M/d")
            )
        )
    )
    .withColumn(
        "release_year",
        F.year(F.col("release_date_parsed"))
    )
)

In [0]:
# Validate release-date parsing
# ---------------------------------------------------------------------------

release_date_check_df = steam_clean_df.agg(
    F.sum(
        F.col("release_date").isNull().cast("int")
    ).alias("missing_source_dates"),
    
    F.sum(
        (
            F.col("release_date").isNotNull()
            & F.col("release_date_parsed").isNull()
        ).cast("int")
    ).alias("parsing_failures"),
    
    F.min("release_date_parsed").alias("earliest_release_date"),
    F.max("release_date_parsed").alias("latest_release_date"),
    F.min("release_year").alias("earliest_release_year"),
    F.max("release_year").alias("latest_release_year"),
)

display(release_date_check_df)

missing_source_dates,parsing_failures,earliest_release_date,latest_release_date,earliest_release_year,latest_release_year
99,123,1997-06-30,2022-11-11,1997,2022


In [0]:
# Inspect parsed release dates
# ---------------------------------------------------------------------------

display(
    steam_clean_df
    .select(
        "appid",
        "name",
        "release_date",
        "release_date_parsed",
        "release_year",
    )
    .orderBy("release_date_parsed")
    .limit(10)
)

appid,name,release_date,release_date_parsed,release_year
1095710,Björk Vulnicura Virtual Reality Album,2019/09,null,null
102500,Kingdoms of Amalur: Reckoning,null,null,null
1069480,Worldwide Sports Fishing,2019/05,null,null
1005930,Timeflow – Life Sim,2019/01,null,null
1048790,Phantasmata,2019/04,null,null
1044130,不惑英雄传(puzzled heroes),2019/04,null,null
1106880,Tribes of Midgard - Open Beta,2019/07,null,null
1118840,My name is You and it's the only unusual thing in my life,2019/12,null,null
1075950,Disintegration Technical Beta,null,null,null
1124010,Princesses vs Dragons: Royal Rumble,2019/08,null,null


**Date-format observation.** Most release dates follow the expected `yyyy/M/d` pattern, but some records contain incomplete values such as `2019/01`. A tolerant parsing strategy is therefore used so that malformed or incomplete dates are converted to null rather than interrupting the pipeline. These records are retained in the dataset and excluded only from analyses that require a valid release date.

### 4.2 Release Date Validation — Key Findings

Release-date preparation produced valid dates ranging from **1997-06-30 to 2022-11-11**, corresponding to release years from **1997 to 2022**.

The source contains **99 missing release dates** and an additional **123 non-empty values that cannot be parsed as complete dates**. Inspection confirms that several of these values are incomplete year-month representations, such as `2019/01`, `2019/05`, or `2019/09`, rather than complete calendar dates.

These observations represent only a small fraction of the 55,691 records. Rather than assigning an arbitrary day to incomplete dates, which would introduce unsupported information, incomplete or malformed dates are retained as null in the parsed date variable.

Games without a valid parsed date remain available for analyses that do not depend on release timing and are excluded only from analyses requiring a complete release date.

### 4.3 Price, Discount, and Free-to-Play Preparation

The source stores initial and current prices as string-encoded monetary values in minor currency units. These variables are converted to numerical values and divided by 100 to obtain standard price units suitable for analysis.

The discount variable has already been converted to a numerical percentage.

Two additional indicators are created:

- `is_free`: identifies games whose initial price is zero;
- `is_discounted`: identifies games with a strictly positive discount percentage.

Using the initial price for the free-game indicator avoids incorrectly classifying a temporarily discounted paid game as inherently free-to-play.

In [0]:
# Convert prices and derive pricing indicators
# ---------------------------------------------------------------------------

steam_clean_df = (
    steam_clean_df
    .withColumn(
        "initial_price",
        F.expr("try_cast(initial_price_raw as double)") / 100
    )
    .withColumn(
        "current_price",
        F.expr("try_cast(price_raw as double)") / 100
    )
    .withColumn(
        "is_free",
        F.col("initial_price") == 0
    )
    .withColumn(
        "is_discounted",
        F.col("discount_pct") > 0
    )
)

In [0]:
# Validate price and discount transformations
# ---------------------------------------------------------------------------

price_check_df = steam_clean_df.agg(
    F.sum(
        F.col("initial_price").isNull().cast("int")
    ).alias("invalid_initial_price"),

    F.sum(
        F.col("current_price").isNull().cast("int")
    ).alias("invalid_current_price"),

    F.sum(
        (F.col("initial_price") < 0).cast("int")
    ).alias("negative_initial_prices"),

    F.sum(
        (F.col("current_price") < 0).cast("int")
    ).alias("negative_current_prices"),

    F.min("initial_price").alias("min_initial_price"),
    F.max("initial_price").alias("max_initial_price"),

    F.min("current_price").alias("min_current_price"),
    F.max("current_price").alias("max_current_price"),

    F.min("discount_pct").alias("min_discount_pct"),
    F.max("discount_pct").alias("max_discount_pct"),

    F.sum(
        F.col("is_free").cast("int")
    ).alias("free_games"),

    F.sum(
        F.col("is_discounted").cast("int")
    ).alias("discounted_games"),
)

display(price_check_df)

invalid_initial_price,invalid_current_price,negative_initial_prices,negative_current_prices,min_initial_price,max_initial_price,min_current_price,max_current_price,min_discount_pct,max_discount_pct,free_games,discounted_games
0,0,0,0,0.0,999.0,0.0,999.0,0.0,90.0,7780,2518


In [0]:
# Inspect the highest-priced games
# ---------------------------------------------------------------------------

display(
    steam_clean_df
    .select(
        "appid",
        "name",
        "initial_price_raw",
        "price_raw",
        "initial_price",
        "current_price",
        "discount_pct",
    )
    .orderBy(F.desc("initial_price"))
    .limit(10)
)

appid,name,initial_price_raw,price_raw,initial_price,current_price,discount_pct
1200520,Ascent Free-Roaming VR Experience,99900,99900,999.0,999.0,0.0
253670,Aartform Curvy 3D 3.0,29990,29990,299.9,299.9,0.0
502570,Houdini Indie,26999,26999,269.99,269.99,0.0
2070990,VEGAS Edit 20 Steam Edition,24900,24900,249.0,249.0,0.0
1259300,Spot Sample Witness Simulator,19999,19999,199.99,199.99,0.0
1429800,Chandrayaan VR,19999,19999,199.99,199.99,0.0
1035340,眼睛（眼球）结构研究,19999,19999,199.99,199.99,0.0
1538090,Virtual Orator,19999,19999,199.99,199.99,0.0
1022640,Lgnorant girl doll,19999,19999,199.99,199.99,0.0
1103060,Run Thief,19999,19999,199.99,199.99,0.0


### 4.3 Price and Discount Validation — Key Findings

All **55,691 price records** were successfully converted from their source string representation to numerical standard price units. No invalid or negative initial or current prices were detected.

Initial and current prices range from **0 to 999**, while observed discounts range from **0% to 90%**. The dataset contains **7,780 games with an initial price of zero** and **2,518 currently discounted games**.

Inspection of the highest-priced observations confirms that the source-to-standard-unit conversion is internally consistent. For example, a raw value of `29990` becomes `299.90`, while `99900` becomes `999.00`.

The highest prices are substantially above typical videogame price levels and include specialized software or VR-related products. However, there is no evidence that these observations result from conversion errors. They are therefore preserved as legitimate extreme observations rather than being removed as outliers.

Price distributions and the potential influence of extreme observations will be examined during the exploratory analysis, where robust statistics such as the median can be used alongside averages when appropriate.

### 4.4 Review and Popularity Features

The dataset does not provide a direct measure of game revenue or commercial success. Player-review activity is therefore used as an observable indicator of market engagement, while concurrent users provide an additional activity measure.

Two review features are derived:

- `total_reviews`, representing the total number of positive and negative reviews;
- `positive_review_ratio`, representing the share of reviews that are positive.

The positive-review ratio is calculated only for games with at least one review. Games with zero reviews are assigned a null ratio rather than an artificial score of zero, since the absence of reviews does not imply negative reception.

Review volume and concurrent-user counts will be interpreted as **engagement or popularity proxies**, not as direct measures of sales or profitability.

In [0]:
# Derive review-based analytical features
# ---------------------------------------------------------------------------

steam_clean_df = (
    steam_clean_df
    .withColumn(
        "total_reviews",
        F.col("positive_reviews") + F.col("negative_reviews")
    )
    .withColumn(
        "positive_review_ratio",
        F.when(
            F.col("total_reviews") > 0,
            F.col("positive_reviews") / F.col("total_reviews")
        )
    )
)

In [0]:
# Validate review features
# ---------------------------------------------------------------------------

review_check_df = steam_clean_df.agg(
    F.min("total_reviews").alias("min_total_reviews"),
    F.max("total_reviews").alias("max_total_reviews"),
    F.sum(
        (F.col("total_reviews") == 0).cast("int")
    ).alias("games_without_reviews"),
    F.min("positive_review_ratio").alias("min_positive_ratio"),
    F.max("positive_review_ratio").alias("max_positive_ratio"),
)

display(review_check_df)


# Check the Steam application types represented in the dataset
# ---------------------------------------------------------------------------

display(
    steam_clean_df
    .groupBy("type")
    .agg(F.count("*").alias("game_count"))
    .orderBy(F.desc("game_count"))
)

min_total_reviews,max_total_reviews,games_without_reviews,min_positive_ratio,max_positive_ratio
0,6730438,163,0.0,1.0


type,game_count
game,55690
hardware,1


### 4.4 Review Feature Validation — Key Findings

The derived review features are internally consistent. Total review counts range from **0 to 6,730,438**, and the positive-review ratio remains within its expected theoretical range of **0 to 1**.

Only **163 games** have no recorded reviews. For these observations, the positive-review ratio remains null because no player-reception score can be inferred from the absence of reviews.

The maximum review count is substantially larger than most observations but is not treated as a data-quality error. Highly popular Steam titles can accumulate very large review volumes, so extreme values are preserved and will be handled appropriately during exploratory analysis through suitable summary statistics or visualization scales.

The application-type check identified **55,690 games and one hardware record**. Because the project focuses specifically on the videogame market, the single hardware observation is excluded from the analytical dataset.

Review volume and concurrent-user activity remain interpreted as indicators of engagement or popularity rather than direct measures of sales or profitability.

In [0]:
# Retain videogame applications only
# ---------------------------------------------------------------------------

steam_clean_df = (
    steam_clean_df
    .filter(F.col("type") == "game")
    .drop("type")
)

In [0]:
# Verify the final game-level record count
# ---------------------------------------------------------------------------

print(f"Number of videogames retained: {steam_clean_df.count():,}")

Number of videogames retained: 55,690


### 4.5 Ownership Range Preparation

Steam ownership information is provided as a range rather than as an exact number of owners. The lower and upper bounds are therefore extracted into numerical variables.

A midpoint is also calculated as a convenient approximate ownership indicator for selected aggregate and multivariate analyses.

The midpoint must not be interpreted as an exact sales figure. Ownership ranges are approximate and do not provide information about purchase price, refunds, free acquisitions, or revenue. Ownership-derived variables are therefore used only as **commercial-reach or popularity proxies**.

In [0]:
# Parse ownership ranges
# ---------------------------------------------------------------------------

steam_clean_df = (
    steam_clean_df
    .withColumn(
        "owners_clean",
        F.regexp_replace(F.col("owners_raw"), ",", "")
    )
    .withColumn(
        "owner_lower_bound",
        F.trim(
            F.split(F.col("owners_clean"), r"\.\.").getItem(0)
        ).cast("long")
    )
    .withColumn(
        "owner_upper_bound",
        F.trim(
            F.split(F.col("owners_clean"), r"\.\.").getItem(1)
        ).cast("long")
    )
    .withColumn(
        "owner_midpoint",
        (
            F.col("owner_lower_bound")
            + F.col("owner_upper_bound")
        ) / 2
    )
    .drop("owners_clean")
)

In [0]:
# Validate ownership-range parsing
# ---------------------------------------------------------------------------

ownership_check_df = steam_clean_df.agg(
    F.sum(
        F.col("owner_lower_bound").isNull().cast("int")
    ).alias("invalid_lower_bound"),

    F.sum(
        F.col("owner_upper_bound").isNull().cast("int")
    ).alias("invalid_upper_bound"),

    F.min("owner_lower_bound").alias("min_lower_bound"),
    F.max("owner_upper_bound").alias("max_upper_bound"),
)

display(ownership_check_df)

invalid_lower_bound,invalid_upper_bound,min_lower_bound,max_upper_bound
0,0,0,500000000


### 4.5 Ownership Validation — Key Findings

Ownership-range parsing was successful for all retained videogame records, with no invalid lower or upper bounds detected.

The ownership ranges extend from a lower bound of **0** to a maximum upper bound of **500 million owners**. Large ownership values are preserved because they may legitimately correspond to exceptionally widely distributed Steam titles.

Because the source provides ownership intervals rather than exact counts, the derived midpoint is treated only as an approximate indicator of commercial reach or popularity. It is not interpreted as an exact number of owners, sales, or revenue.

### 4.6 Genre, Language, and Platform Features

Genres and languages are stored as comma-separated strings in the source dataset. They are converted into arrays so that they can be used efficiently in subsequent Spark transformations.

At the game level, the number of genres and supported languages is derived from these arrays. Genres will not be exploded in this preparation notebook; the explosion will be performed later when constructing the dedicated genre-level analytical DataFrame.

Platform availability is already represented by Boolean indicators for Windows, macOS, and Linux. A `platform_count` feature is created to distinguish single-platform from multi-platform availability while preserving the individual platform indicators.

These transformations create reusable analytical features without unnecessarily changing the game-level granularity of the prepared dataset.

In [0]:
# Prepare genre, language, and platform features
# ---------------------------------------------------------------------------

steam_clean_df = (
    steam_clean_df

    # Convert comma-separated genres to a cleaned array
    .withColumn(
        "genres",
        F.when(
            F.col("genres_raw").isNotNull(),
            F.transform(
                F.split(F.col("genres_raw"), ","),
                lambda x: F.trim(x)
            )
        )
    )

    # Convert comma-separated languages to a cleaned array
    .withColumn(
        "languages",
        F.when(
            F.col("languages_raw").isNotNull(),
            F.transform(
                F.split(F.col("languages_raw"), ","),
                lambda x: F.trim(x)
            )
        )
    )

    # Derive game-level counts
    .withColumn(
        "genre_count",
        F.when(
            F.col("genres").isNotNull(),
            F.size(F.col("genres"))
        )
    )
    .withColumn(
        "language_count",
        F.when(
            F.col("languages").isNotNull(),
            F.size(F.col("languages"))
        )
    )

    # Count supported desktop platforms
    .withColumn(
        "platform_count",
        F.col("supports_windows").cast("int")
        + F.col("supports_mac").cast("int")
        + F.col("supports_linux").cast("int")
    )
)

In [0]:
# Validate genre, language, and platform features
# ---------------------------------------------------------------------------

feature_check_df = steam_clean_df.agg(
    F.sum(F.col("genres").isNull().cast("int")).alias("missing_genres"),
    F.min("genre_count").alias("min_genre_count"),
    F.max("genre_count").alias("max_genre_count"),

    F.sum(F.col("languages").isNull().cast("int")).alias("missing_languages"),
    F.min("language_count").alias("min_language_count"),
    F.max("language_count").alias("max_language_count"),

    F.min("platform_count").alias("min_platform_count"),
    F.max("platform_count").alias("max_platform_count"),

    F.sum(F.col("supports_windows").cast("int")).alias("windows_games"),
    F.sum(F.col("supports_mac").cast("int")).alias("mac_games"),
    F.sum(F.col("supports_linux").cast("int")).alias("linux_games"),
)

display(feature_check_df)

missing_genres,min_genre_count,max_genre_count,missing_languages,min_language_count,max_language_count,min_platform_count,max_platform_count,windows_games,mac_games,linux_games
160,1,16,10,1,40,1,3,55675,12769,8457


### 4.6 Genre, Language, and Platform Validation — Key Findings

The derived genre, language, and platform features are structurally consistent across the retained videogame dataset.

After excluding the single hardware record, **160 games have no genre information** and **10 games have no language information**. Games with available genre data contain between **1 and 16 genres**, while games with language information support between **1 and 40 languages**.

Every retained videogame supports at least one of the three recorded desktop platforms. Platform availability ranges from **1 to 3 supported platforms**, so no zero-platform observations require additional cleaning.

Windows is supported by **55,675 games**, compared with **12,769 games on macOS** and **8,457 games on Linux**. These counts are not mutually exclusive because the same game can support multiple platforms.

The cleaned arrays and derived counts preserve the original game-level granularity. Genre explosion and platform comparisons will be performed later in the dedicated exploratory analyses, where they directly support the business questions.

### 4.7 Final Analytical Dataset Validation

A final validation is performed after cleaning and feature preparation to confirm the structure of the game-level analytical dataset.

The validation checks the final record count, identifier uniqueness, and selected derived variables. This provides a reproducible checkpoint before the prepared data is used for market, genre, and platform analysis.

At this stage, remaining null values are expected and documented. They represent unavailable or incomplete source information rather than unresolved processing errors.

In [0]:
# Final dataset integrity check
# ---------------------------------------------------------------------------

final_validation_df = steam_clean_df.agg(
    F.count("*").alias("total_games"),
    F.countDistinct("appid").alias("distinct_appids"),
    F.sum(F.col("publisher").isNull().cast("int")).alias("missing_publisher"),
    F.sum(F.col("release_date_parsed").isNull().cast("int")).alias("missing_parsed_date"),
    F.sum(F.col("genres").isNull().cast("int")).alias("missing_genres"),
    F.sum(F.col("languages").isNull().cast("int")).alias("missing_languages"),
    F.sum(F.col("required_age").isNull().cast("int")).alias("missing_numeric_age"),
)

display(final_validation_df)

total_games,distinct_appids,missing_publisher,missing_parsed_date,missing_genres,missing_languages,missing_numeric_age
55690,55690,134,222,160,10,3


In [0]:
# Inspect the final analytical schema
# ---------------------------------------------------------------------------

steam_clean_df.printSchema()

root
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- initial_price_raw: string (nullable = true)
 |-- price_raw: string (nullable = true)
 |-- discount_raw: string (nullable = true)
 |-- positive_reviews: long (nullable = true)
 |-- negative_reviews: long (nullable = true)
 |-- concurrent_users: long (nullable = true)
 |-- owners_raw: string (nullable = true)
 |-- genres_raw: string (nullable = true)
 |-- languages_raw: string (nullable = true)
 |-- required_age_raw: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- supports_windows: boolean (nullable = true)
 |-- supports_mac: boolean (nullable = true)
 |-- supports_linux: boolean (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- required_age: integer (nullable = true)
 |-- release_date_parsed: date

In [0]:
# Preview the prepared game-level dataset
# ---------------------------------------------------------------------------

display(
    steam_clean_df.select(
        "appid",
        "name",
        "publisher",
        "release_year",
        "initial_price",
        "current_price",
        "discount_pct",
        "is_free",
        "total_reviews",
        "positive_review_ratio",
        "owner_midpoint",
        "genres",
        "language_count",
        "required_age",
        "supports_windows",
        "supports_mac",
        "supports_linux",
        "platform_count",
    ).limit(10)
)

appid,name,publisher,release_year,initial_price,current_price,discount_pct,is_free,total_reviews,positive_review_ratio,owner_midpoint,genres,language_count,required_age,supports_windows,supports_mac,supports_linux,platform_count
10,Counter-Strike,Valve,2000,9.99,9.99,0.0,false,206414,0.9748127549487923,1.5E7,List(Action),8,0,true,true,true,3
1000000,ASCENXION,PsychoFlux Entertainment,2021,9.99,9.99,0.0,false,32,0.84375,10000.0,"List(Action, Adventure, Indie)",3,0,true,false,false,1
1000010,Crown Trick,"Team17, NEXT Studios",2020,19.99,5.99,70.0,false,4678,0.8619067977768277,350000.0,"List(Adventure, Indie, RPG, Strategy)",9,0,true,false,false,1
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,2020,19.99,19.99,0.0,false,1690,0.9319526627218935,150000.0,"List(Action, Indie, Simulation, Strategy)",1,0,true,true,false,2
1000040,细胞战争,DoubleC Games,2019,1.99,1.99,0.0,false,1,0.0,10000.0,"List(Action, Casual, Indie, Simulation)",1,0,true,false,false,1
1000080,Zengeon,2P Games,2019,19.99,7.99,60.0,false,1480,0.6878378378378378,150000.0,"List(Action, Adventure, Indie, RPG)",5,0,true,true,false,2
1000100,干支セトラ 陽ノ卷｜干支etc. 陽之卷,Starship Studio,2019,12.99,12.99,0.0,false,24,0.75,10000.0,"List(Adventure, Indie, RPG, Strategy)",3,0,true,false,false,1
1000110,Jumping Master(跳跳大咖),重庆环游者网络科技,2019,0.0,0.0,0.0,true,84,0.5952380952380952,35000.0,"List(Action, Adventure, Casual, Free to Play, Massively Multiplayer)",3,0,true,false,false,1
1000130,Cube Defender,Simon Codrington,2019,2.99,2.99,0.0,false,6,1.0,10000.0,"List(Casual, Indie)",1,0,true,true,false,2
1000280,Tower of Origin2-Worm's Nest,Villain Role,2021,13.99,13.99,0.0,false,44,0.7272727272727273,10000.0,"List(Indie, RPG)",3,0,true,false,false,1


## 5. Prepared Dataset Summary

The data-preparation pipeline produces a validated game-level Spark DataFrame containing **55,690 unique videogames**.

The original nested JSON structure has been selectively transformed into analytical variables while preserving the game-level granularity of the dataset. The preparation process included:

- validation and consolidation of the Steam application identifier;
- standardization of empty-string missing values;
- exclusion of one non-videogame hardware record;
- safe parsing of release dates and extraction of release year;
- conversion of price, discount, and age variables to analytical numerical types;
- creation of free-to-play and discount indicators;
- derivation of total review volume and positive-review ratio;
- extraction of ownership-range bounds and an approximate ownership midpoint;
- conversion of genre and language strings into reusable arrays;
- derivation of genre, language, and platform counts.

### Final Data Quality

The final dataset contains **55,690 records and 55,690 distinct application identifiers**, confirming that the game-level uniqueness constraint is preserved.

A small amount of missing information remains:

- 134 games have no publisher information;
- 222 games have no complete parseable release date;
- 160 games have no genre information;
- 10 games have no language information;
- 3 games contain age-rating information that cannot be represented as a numerical minimum age.

These observations are retained whenever possible and will be excluded only from analyses requiring the corresponding variable.

Extreme values in price, review volume, and ownership are also preserved because the validation process did not identify them as transformation errors. Their influence will instead be considered during exploratory analysis using appropriate aggregations and robust summary statistics.

### Analytical Dataset

The prepared DataFrame now contains the principal dimensions required for the subsequent analysis:

**Market characteristics**
- publisher and developer;
- release date and release year;
- initial and current price;
- discount percentage;
- free-to-play and discount indicators;
- language availability;
- age requirements.

**Reception and popularity proxies**
- positive and negative reviews;
- total review volume;
- positive-review ratio;
- concurrent users;
- ownership-range estimates.

**Product characteristics**
- genres;
- categories;
- Windows, macOS, and Linux availability;
- genre, language, and platform counts.

The dataset is therefore ready for distributed exploratory analysis of the Steam videogame market.

---

## Next Steps

The following analyses will focus on two complementary perspectives:

1. **Steam Market Analysis** — market evolution, publishers, pricing, discounts, languages, age restrictions, reviews, and popularity indicators;
2. **Genre and Platform Analysis** — genre representation and performance, platform availability, cross-platform patterns, and selected multivariate relationships.

Spark remains the primary processing engine. Aggregations will be performed on distributed DataFrames before compact results are used for Databricks visualizations and business interpretation.

In [0]:
# Final prepared game-level analytical DataFrame
# ---------------------------------------------------------------------------

steam_games_df = steam_clean_df

print(f"Prepared videogames: {steam_games_df.count():,}")
print(f"Analytical columns: {len(steam_games_df.columns)}")

Prepared videogames: 55,690
Analytical columns: 37


### Saving the Prepared Dataset

The validated game-level DataFrame is persisted as a Delta table so that subsequent notebooks can access the same prepared dataset without repeating the complete data-cleaning pipeline.

This creates a reproducible handoff between data preparation and exploratory analysis while keeping the subsequent notebooks focused on their analytical objectives.

In [0]:
# Persist the prepared game-level dataset for subsequent notebooks
# ---------------------------------------------------------------------------

TABLE_NAME = "steam_games_prepared"

(
    steam_games_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLE_NAME)
)

print(f"Prepared dataset saved as Delta table: {TABLE_NAME}")

Prepared dataset saved as Delta table: steam_games_prepared


In [0]:
# Verify the persisted analytical dataset
# ---------------------------------------------------------------------------

steam_saved_df = spark.table(TABLE_NAME)

print(f"Saved records: {steam_saved_df.count():,}")
print(f"Saved columns: {len(steam_saved_df.columns)}")

Saved records: 55,690
Saved columns: 37
